In [0]:
CREATE OR REPLACE MATERIALIZED VIEW gshen_catalog.enbridge_sr_workshop.incidents_ai_enriched AS
SELECT
  /* ============================================================
       Bring forward all base incident fields
       ============================================================ */
  b.*,
  /* ============================================================
       AI CLASSIFICATION FOR BENCHMARKING
       Purpose:
         - Standardize inconsistent accident descriptions
         - Enable fair operator-to-operator comparison
       ============================================================ */
  ai_classify(
    CONCAT(
      'Details: ',
      COALESCE(b.ACCIDENT_DETAILS, 'Details not provided'),
      '. ',
      'Component: ',
      COALESCE(b.SYSTEM_PART_INVOLVED, 'Unknown'),
      '. ',
      'Commodity: ',
      COALESCE(b.COMMODITY_RELEASED_TYPE, 'Unknown')
    ),
    ARRAY(
      'Corrosion',
      'Equipment Failure',
      'Incorrect Operation',
      'Natural Forces',
      'Third-Party Damage',
      'Other'
    )
  ) AS ai_benchmark_cause,
  /* ============================================================
       AI EXTRACTION FOR BENCHMARKING
       Purpose:
         - Standardize component categories
         - Enables component-level reliability benchmarking:
               • pipeline vs valve vs pump vs tank
               • incident rates per 1,000 miles by component
       ============================================================ */
  ai_extract(
    COALESCE(b.NARRATIVE, 'No narrative provided'),
    ARRAY(
      'failure_mechanism', -- e.g., mechanical failure of threaded nipple
      'failed_component', -- e.g., drain line, trap, flange, coupling
      'facility_context', -- e.g., pump station, incoming trap, crossover
      'leak_detection_method', -- e.g., alarm, pressure drop, field observation
      'operational_actions', -- e.g., shutdown, isolation, dispatch
      'response_time_minutes', -- AI inference from timestamps
      'estimated_volume_from_narrative', -- e.g., “5 barrels”
      'cause_hint' -- supplemental textual cue for AI classification
    )
  ) AS ai_narrative_features
FROM
  gshen_catalog.enbridge_sr_workshop.incidents_base b
WHERE
  operator_name IN (
    /* ============================
       ENBRIDGE GROUP (ALL LIQUIDS)
       ============================ */
    'ENBRIDGE ENERGY, LIMITED PARTNERSHIP',
    'ENBRIDGE PIPELINES (EAST TEXAS) L.P.',
    'ENBRIDGE PIPELINES (NORTH DAKOTA) LLC',
    'ENBRIDGE PIPELINES (OZARK) L.L.C.',
    'ENBRIDGE PIPELINES (SOUTHERN LIGHTS) L.L.C.',
    'ENBRIDGE PIPELINES (TOLEDO) INC',
    'ENBRIDGE STORAGE (CUSHING) L.L.C.',
    'ENBRIDGE STORAGE (PATOKA) L.L.C.',
    'ENBRIDGE HOLDINGS (GRAY OAK) LLC',
    'ENBRIDGE INGLESIDE, LLC',
    'ENBRIDGE ENERGY MARKETING LLC',
    /* ============================
       PEER 1 — PLAINS PIPELINE
       ============================ */
    'PLAINS PIPELINE, L.P.',
    'PLAINS PIPELINE MIDCON LLC',
    'PLAINS MARKETING, L.P.',
    /* ============================
       PEER 2 — KINDER MORGAN
       ============================ */
    'KINDER MORGAN CRUDE AND CONDENSATE LLC',
    'KINDER MORGAN CO2 CO. LLC',
    'KINDER MORGAN ENERGY PARTNERS, L.P.',
    'KINDER MORGAN PIPELINES (USA) INC',
    'KINDER MORGAN UTOPIA LLC',
    'KINDER MORGAN WINK PIPELINE LLC'
  )
ORDER BY
  incident_year DESC;